In [1]:
quiet_library <- function(...){suppressPackageStartupMessages(library(...))}
quiet_library(dplyr)
quiet_library(purrr)
quiet_library(tidyr)
quiet_library(data.table)
quiet_library(Seurat)
quiet_library(SeuratDisk)
quiet_library(ggplot2)
quiet_library(glue)
quiet_library(hise)
quiet_library(H5weaver)
quiet_library(gridExtra)

options(repr.matrix.max.cols=150, repr.matrix.max.rows=200, mc.cores = 10, future.globals.maxSize = 2000 * 1024^2)
fig.size <- function (height, width) {
    options(repr.plot.height = height, repr.plot.width = width)
}

In [2]:
set.seed(123)

In [3]:
wd <- "/home/workspace/IFN" 

In [ ]:
# Download cell type objects
setwd(file.path(wd, "cache"))
file_uuid <- list("13b924f3-58e2-4222-8fa9-7e61f3fd039f", "62f823db-f026-45a9-917c-2c0c9eece222", 
                  "78830d2e-e279-469e-b850-e34f52715b5f", "8594fddd-a17e-4246-a00f-63c5fc1e8cd8")
hise_res <- cacheFiles(file_uuid)

In [5]:
stims <- c("IFNa", "IFNb", "IFNg", "IFN-L1")
donors <- c("3955BW", "3283BW", "2616BW", "6811BW", "3491BW")
l1_celltypes <- c("Monocyte", "NK", "Tcell", "Bcell")
l2_celltypes <- c("Monocyte", "B_Naive", "B_Memory", "Plasma", 
             "CD4_Naive", "CD4_Memory", "CD8_Naive", "CD8_Memory", "gdT", "Treg", "MAIT", "NK.CD56dim", "NK.CD56hi")

### N=1 downsample MAST

#### L1

In [3]:
celltype_level <- "L1"

In [ ]:
for (c in l1_celltypes){
    dir.create(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c))
    
    so <- readRDS(file.path(wd, "seurat_objects", "celltype_objects", glue("{c}_so_cleaned_v2.rds"))) 
           
    Idents(so) <- "Stim"
    for (i in stims){    
        for (d in donors){    
        sub <- so %>% subset(DonorID == d & Stim %in% c(i, "none"))    
        sub$CDR <- scale(sub$nFeature_RNA)
            
        parallel::mclapply(1:20, function(n){
            set.seed(n)
            # downsample stim + unstim group if both > 1000 cells
            if (all(table(sub$Stim)[c(i, "none")] > 1000)){
                cells_sample <- c(sub@meta.data %>% filter(Stim == i) %>% rownames() %>% sample(1000),
                              sub@meta.data %>% filter(Stim == "none") %>% rownames() %>% sample(1000))
                subdown <- sub %>% subset(cells = cells_sample)
                } else {
                subdown <- sub
                }

            # run MAST with CDR
            degs <- FindMarkers(subdown, ident.1 = i, ident.2 = "none",  
                                group.by = "Stim", test.use = "MAST", logfc.threshold = 0, latent.vars = "CDR", min.pct = 0.01)
            degs$gene <- rownames(degs)
            degs$stim <- i
            degs <- degs[,c("gene", "stim", colnames(degs)[1:5])]
            degs %>% 
                arrange(-abs(avg_log2FC)) %>%
                fwrite(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c,
                                 glue("{d}_{i}_{c}_DEGs_MAST_CDR_downsample1000_iteration{n}.csv")))
            }, mc.cores = 5)
           
        }
    }
}

#### L2

In [ ]:
celltype_level <- "L2"

In [ ]:
for (c in l1_celltypes){
    
    so <- readRDS(file.path(wd, "seurat_objects", "celltype_objects", glue("{c}_so_cleaned_v2.rds"))) 
    l2_celltypes_select <- unique(so$celltype.l2)
    
    for (s in l2_celltypes_select){
        
        dir.create(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, s))
        sub_celltype <- so %>% subset(celltype.l2 == s)
        Idents(so) <- "Stim"
        for (i in stims){    
            for (d in donors){    
            sub <- sub_celltype %>% subset(DonorID == d & Stim %in% c(i, "none"))    
            sub$CDR <- scale(sub$nFeature_RNA)
                
            parallel::mclapply(1:20, function(n){
                set.seed(n)
                # downsample stim + unstim group if both > 1000 cells
                if (all(table(sub$Stim)[c(i, "none")] > 1000)){
                    cells_sample <- c(sub@meta.data %>% filter(Stim == i) %>% rownames() %>% sample(1000),
                                  sub@meta.data %>% filter(Stim == "none") %>% rownames() %>% sample(1000))
                    subdown <- sub %>% subset(cells = cells_sample)
                    } else {
                    subdown <- sub
                    }
    
                # run MAST with CDR
                degs <- FindMarkers(subdown, ident.1 = i, ident.2 = "none",  
                                    group.by = "Stim", test.use = "MAST", logfc.threshold = 0, latent.vars = "CDR", min.pct = 0.01)
                degs$gene <- rownames(degs)
                degs$stim <- i
                degs <- degs[,c("gene", "stim", colnames(degs)[1:5])]
                degs %>% 
                    arrange(-abs(avg_log2FC)) %>%
                    fwrite(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, s,
                                     glue("{d}_{i}_{c}_DEGs_MAST_CDR_downsample1000_iteration{n}.csv")))
                }, mc.cores = 5)
               
            }
        }
    }
}

### Per donor, get consensus set at 10 iterations of downsampling

In [6]:
if (celltype_level == "L1"){celltypes <- l1_celltypes}
if (celltype_level == "L2"){celltypes <- l2_celltypes}

In [14]:
for (c in celltypes){
    for (i in stims){
        for (d in donors){
                # read in donor level results
                degs_all <- map_dfr(1:20, function(n){          
                            # run MAST DEG
                            degs <- fread(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c, 
                                                 glue("{d}_{i}_{c}_DEGs_MAST_CDR_downsample1000_iteration{n}.csv"))) 
                            degs$iteration <- n
                            # define significance threshold 
                            degs$Sig <- ifelse(degs$p_val_adj < 0.01 & abs(degs$avg_log2FC) > 0.2, "Yes", "No")
                            degs

                        })

                    # retain genes significant across 10 iterations 
                    degs_all <- degs_all %>%
                          group_by(gene) %>%
                          mutate(nSig = sum(Sig == "Yes")) %>% 
                          mutate(Pass_Consensus = ifelse(nSig >= 10, "Yes", "No")) %>%
                          mutate(iteration_median_log2FC = median(avg_log2FC)) %>% 
                          mutate(iteration_median_pval = median(p_val_adj))
                
                    degs_all %>% 
                            select(gene, stim, iteration_median_log2FC, iteration_median_pval, Pass_Consensus) %>%
                            unique() %>% 
                            fwrite(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c,
                                          glue("{d}_{i}_{c}_DEGs_MAST_CDR_downsample1000_consensus.csv"))) 
             
            }
        }
}



### Create consensus DEG lists

In [ ]:
for (c in celltypes){
     for (i in stims){ 
            degs_all <- map_dfr(donors, function(d){   
                        degs <- fread(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c,
                                                glue("{d}_{i}_{c}_DEGs_MAST_CDR_downsample1000_consensus.csv"))) 
                        degs$donor <- d

                        if (nrow(degs) > 0){
                            degs
                            }
                    })

                    # mark genes significant across all five donors and generate summary statistics across donors
                    degs_consensus <- degs_all %>% 
                        group_by(gene) %>% 
                        mutate(nDonors_Sig = sum(Pass_Consensus == "Yes")) %>% 
                        mutate(same_sign = all(sign(iteration_median_log2FC) == sign(iteration_median_log2FC[1]))) %>%
                        summarise(median_log2FC = median(iteration_median_log2FC),
                                 median_pval = median(iteration_median_pval),
                                 max_pval = max(iteration_median_pval),
                                  min_log2FC = iteration_median_log2FC[which.min(abs(iteration_median_log2FC))],
                                 significant = ifelse(nDonors_Sig == 5 & same_sign == TRUE, "Yes", "No"),
                                 celltype = c) %>%
                        unique() %>% 
                        arrange(-abs(median_log2FC))

                    degs_consensus %>% fwrite(file.path(wd, "DEGs", "MAST_N1_downsample", celltype_level, c, 
                                                        glue("Consensus_{i}_{c}_DEGs_MAST_Significant_MaxP.csv")))

    }
}


### Upload

In [ ]:
out_files <- list.files(file.path(wd, "DEGs", "MAST_N1_downsample", "L1"), recursive = T, full.names = T) %>% 
                grep("DEGs_MAST_Significant_MaxP", ., value = T) 

In [ ]:
title <- "L1_N1_differentials"

In [ ]:
uploadFiles(
    studySpaceId = study_space_uuid,
    title = title,
    files = out_files,
    inputFileIds = file_uuid
)